# Cross-Domain Object Tracker - Quick Start

This notebook walks you through the core workflow of **crossdomain-object-tracker**:
downloading multi-domain datasets, running object detection, evaluating cross-domain performance, and generating reports.

---

このノートブックでは **crossdomain-object-tracker** の基本的なワークフローを紹介します。
複数ドメインのデータセットのダウンロード、物体検出の実行、ドメイン間の評価、およびレポート生成を行います。

In [ ]:
# !pip install -e "..[dev]"

## 1. Download Sample Data

In [ ]:
from crossdomain_object_tracker.common.download import download_dataset
from crossdomain_object_tracker.common.config import get_project_root

data_dir = get_project_root() / "data"

# Download demo samples for each dataset
for dataset in ["covla", "polaris", "mcd", "hm3d_ovon"]:
    download_dataset(dataset, output_dir=str(data_dir / dataset), demo=True)

## 2. Run Object Detection (YOLOv8)

In [ ]:
from crossdomain_object_tracker.detector import get_detector, Detection
from pathlib import Path
import glob

detector = get_detector("yolov8n")

# Detect on one sample image
sample_images = list(Path(data_dir / "covla").rglob("*.jpg")) + list(Path(data_dir / "covla").rglob("*.png"))
if sample_images:
    detections = detector.detect(str(sample_images[0]))
    print(f"Found {len(detections)} objects:")
    for d in detections[:5]:
        print(f"  {d.class_name}: {d.confidence:.2f} at {d.bbox}")

## 3. Cross-Domain Evaluation

In [ ]:
from crossdomain_object_tracker.evaluate import evaluate_dataset, compare_domains

results = {}
for dataset_name in ["covla", "polaris", "mcd", "hm3d_ovon"]:
    dataset_dir = data_dir / dataset_name
    if dataset_dir.exists() and any(dataset_dir.rglob("*.jpg")) or any(dataset_dir.rglob("*.png")):
        results[dataset_name] = evaluate_dataset(detector, dataset_name, str(dataset_dir), max_samples=10)
        print(f"{dataset_name}: {results[dataset_name].get('total_detections', 0)} detections")

if results:
    comparison = compare_domains(results)
    print("\n=== Domain Comparison ===")
    print(comparison.to_string())

## 4. Visualization

In [ ]:
from crossdomain_object_tracker.visualize import (
    draw_detections, plot_class_distribution,
    plot_confidence_distribution, plot_detection_counts
)
import matplotlib.pyplot as plt

# Draw detections on sample image
if sample_images and detections:
    img = draw_detections(str(sample_images[0]), detections)
    plt.figure(figsize=(12, 8))
    plt.imshow(img[:, :, ::-1])  # BGR to RGB
    plt.axis("off")
    plt.title("YOLOv8 Detections")
    plt.show()

# Compare across domains
if results:
    plot_class_distribution(results)
    plot_confidence_distribution(results)
    plot_detection_counts(results)

## 5. Generate Report

In [ ]:
from crossdomain_object_tracker.report import generate_report

if results:
    output_dir = get_project_root() / "outputs" / "report"
    generate_report(results, str(output_dir))
    print(f"Report generated at: {output_dir}/report.html")

## 6. Open-Vocabulary Detection (Grounding DINO)

This section demonstrates open-vocabulary detection using Grounding DINO.
An additional dependency is required (`pip install -e ".[grounding-dino]"`).

In [ ]:
# Requires: pip install -e ".[grounding-dino]"
try:
    gdino = get_detector("grounding-dino")
    if sample_images:
        detections = gdino.detect(str(sample_images[0]), text_prompt="car . person . bicycle . traffic light")
        print(f"Grounding DINO found {len(detections)} objects:")
        for d in detections[:5]:
            print(f"  {d.class_name}: {d.confidence:.2f}")
except Exception as e:
    print(f"Grounding DINO not available: {e}")
    print("Install with: pip install transformers")

---

## References

- **COVLa**: Multimodal driving dataset with rich annotations
- **Polaris**: All-weather autonomous driving dataset
- **MCD**: Multi-Campus Dataset for cross-domain evaluation
- **HM3D-OVON**: Habitat-Matterport 3D Object-Goal Navigation dataset
- **YOLOv8**: Ultralytics, [https://github.com/ultralytics/ultralytics](https://github.com/ultralytics/ultralytics)
- **Grounding DINO**: Liu et al., "Grounding DINO: Marrying DINO with Grounded Pre-Training for Open-Set Object Detection", 2023